In [ ]:
import numpy as np

F087 = np.load("roman-data-challenge-uiuc/src/filter_data/F087.npy")
F146 = np.load("roman-data-challenge-uiuc/src/filter_data/F146.npy")
F213 = np.load("roman-data-challenge-uiuc/src/filter_data/F213.npy")
print(F087.shape)
print(F146.shape)
print(F146.shape)

# GAME PLAN:
# All 3 filters observe 188 stars. In all 188 stars, we keep track of 3 properties: mag, mag_err, and bjd.
# Filters F146 and F213 have 46208 epochs per star while F087 only has 1445
# We are going to conduct the PSPL (Point-Source Point-Lens) model to find the chi squared
# Then we are going to sort the list of chi squared in descending order to find anomalies
# Lastly, we are going to plot the F146 anomaly data against the fitted PSPL curve, and check whether
# any F087 or F213 points that happen to fall near the anomalous region

(188, 3, 1445)
(188, 3, 46208)
(188, 3, 46208)


In [ ]:
from scipy.optimize import curve_fit
def pspl_mag(t, m0, t0, tE, u0):
    # Returns mag_pspl
    u = np.sqrt(u0**2 + ((t- t0) / tE)**2)
    A = (u**2 + 2) / (u * np.sqrt(u**2 + 4))
    return m0- 2.5 * np.log10(A)

def fit_and_score(t, mag, mag_err, t0_guess, teff_guess, u0_guess=0.5):
    # Returns optimized parameters and chi_squared
    tE_guess = teff_guess / u0_guess
    p0 = [np.median(mag), t0_guess, tE_guess, u0_guess]
    popt, _ = curve_fit(pspl_mag, t, mag, p0=p0, sigma=mag_err, maxfev=5000)
    resid = mag- pspl_mag(t, *popt)
    chi2 = np.sum((resid / mag_err) ** 2) / (len(t)- 4)
    return popt, chi2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tslearn.clustering import KShape

df = pd.read_parquet("roman-data-challenge-uiuc\data\RMDC26_Beginner_Tier_test.parquet")
print(df.columns)
print(df.shape)

event_names = [f"RMDC26_{i:06d}" for i in range(1, 189)]

# event names are from RMDC26_000001 to RMDC26_000188
# filters are ['F087', 'F146', 'F213']

def makeFliterDataMatrix(filter, df):
  res = []
  filter_name = filter
  for event_id in event_names:
    res.append(df.loc[(df["name"] == event_id) & (df["filt"] == filter_name), ["mag","mag_err","bjd"]].sort_values("bjd").to_numpy().T)
  return np.array(res)

F087 = makeFliterDataMatrix("F087", df)
print(F087.shape)
F146 = makeFliterDataMatrix("F146", df)
print(F146.shape)
F213 = makeFliterDataMatrix("F213", df)
print(F146.shape)

<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Mathew Icho\AppData\Local\Temp\ipykernel_4160\396956209.py:6: SyntaxWarning: invalid escape sequence '\d'
  df = pd.read_parquet("roman-data-challenge-uiuc\data\RMDC26_Beginner_Tier_test.parquet")


Index(['name', 'l_deg', 'b_deg', 'ra_deg', 'dec_deg', 'bjd', 'filt', 'mag',
       'mag_err', 'extinction', 'obs_x', 'obs_y', 'obs_z', 'saturation_flag'],
      dtype='object')
(9230048, 14)
(188, 3, 1445)
(188, 3, 46208)
(188, 3, 46208)
